In [1]:
import os

In [2]:
import torch

if torch.cuda.is_available():
    print("GPU is available!")
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU is not available. Please check Runtime settings.")


GPU is not available. Please check Runtime settings.


In [ ]:
from google.colab import files

uploaded = files.upload()


In [ ]:
print("Files currently available in Colab:\n")

for file in os.listdir("/content"):
    print(file)


In [ ]:
import zipfile


# Find the uploaded ZIP file
zip_files = [
    file for file in os.listdir("/content")
    if file.lower().endswith(".zip")
]

if not zip_files:
    raise FileNotFoundError("No ZIP file found. Please upload your dataset.")

zip_path = os.path.join("/content", zip_files[0])

# Create the extraction folder
extract_path = "/content/data"
os.makedirs(extract_path, exist_ok=True)

# Extract the ZIP
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

print("Dataset extracted successfully!")
print("Extracted location:", extract_path)


**Dataset inspection**

In [ ]:
for root, dirs, files_list in os.walk("/content/data"):
    level = root.replace("/content/data", "").count(os.sep)

    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    for file in files_list:
        print(f"{indent}    {file}")


In [ ]:
# Main project folder
PROJECT_DIR = "/content/data"

# Actual dataset folder
DATA_DIR = os.path.join(PROJECT_DIR, "data")

# Individual dataset paths
TRAIN_PATH = os.path.join(DATA_DIR, "train.json")
VAL_PATH = os.path.join(DATA_DIR, "val.json")
TEST_PATH = os.path.join(DATA_DIR, "test.json")

# Supporting files
TOOLS_PATH = os.path.join(DATA_DIR, "tools.json")
STATS_PATH = os.path.join(DATA_DIR, "stats.json")
TAXONOMY_PATH = os.path.join(DATA_DIR, "taxonomy.json")

# Check whether all required files exist
required_paths = [
    TRAIN_PATH,
    VAL_PATH,
    TEST_PATH,
    TOOLS_PATH,
    STATS_PATH,
    TAXONOMY_PATH
]

for path in required_paths:
    print(("FOUND  " if os.path.exists(path) else "MISSING ") + path)


In [ ]:
import json

with open(STATS_PATH, "r", encoding="utf-8") as file:
    dataset_stats = json.load(file)

print(json.dumps(dataset_stats, indent=4))

In [ ]:
with open(TRAIN_PATH, "r", encoding="utf-8") as file:
    train_data = json.load(file)

with open(VAL_PATH, "r", encoding="utf-8") as file:
    val_data = json.load(file)

with open(TEST_PATH, "r", encoding="utf-8") as file:
    test_data = json.load(file)

print("Dataset loading completed!\n")

print("Training conversations:", len(train_data))
print("Validation conversations:", len(val_data))
print("Testing conversations:", len(test_data))
print("Total conversations:",
      len(train_data) + len(val_data) + len(test_data))


In [ ]:
sample = train_data[0]

print("Conversation ID:", sample.get("id"))

print("\nMessage roles:")

for message in sample.get("messages", []):
    print("Role:", message.get("role"))
    print("Content:", str(message.get("content", ""))[:250])
    print("-" * 50)


***Dataset Validation.***

In [ ]:
import os

# Move into the main project directory
os.chdir("/content/data")

# Confirm current working directory
print("Current directory:", os.getcwd())

# Confirm validation script exists
print("validate.py exists:", os.path.exists("validate.py"))


In [ ]:
!python validate.py

In [ ]:

import json

with open("/content/data/data/tools.json", "r", encoding="utf-8") as file:
    tools = json.load(file)

print("Total tools:", len(tools))

for tool in tools:
    function = tool["function"]

    print("\nTool name:", function["name"])
    print("Description:", function.get("description", "No description"))

    parameters = function.get("parameters", {})
    print("Required arguments:", parameters.get("required", []))
    print("Available arguments:", list(parameters.get("properties", {}).keys()))

    print("-" * 60)


In [ ]:
!pip install Unsloth

In [ ]:
import torch
import unsloth

print("Unsloth version:", unsloth.__version__)
print("PyTorch version:", torch.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
import torch
import unsloth

print("Unsloth version:", unsloth.__version__)
print("PyTorch version:", torch.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU is not detected.")

In [ ]:
import torch
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen3-4B-Instruct-2507"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)

print("Model loaded successfully!")

In [ ]:
from unsloth import FastLanguageModel

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

print("LoRA adapters configured successfully!")

In [ ]:
model.print_trainable_parameters()

**Load tool definitions**

In [ ]:
import json
import os

TOOLS_PATH = "/content/data/data/tools.json"
TRAIN_PATH = "/content/data/data/train.json"

# Load tool definitions
with open(TOOLS_PATH, "r") as f:
    tools = json.load(f)

# Load training conversations
with open(TRAIN_PATH, "r") as f:
    train_data = json.load(f)

print("Total tools:", len(tools))
print("Total training conversations:", len(train_data))

# Inspect one conversation
sample = train_data[0]

print("\nConversation ID:", sample["id"])
print("Available fields:", sample.keys())

print("\nFirst 5 messages:")
for message in sample["messages"][:5]:
    print(message)

In [ ]:
formatted_text = tokenizer.apply_chat_template(
    sample["messages"],
    tools=tools,
    tokenize=False,
    add_generation_prompt=False
)

print(formatted_text[:4000])

In [ ]:
# Test assistant-token masking on one training conversation

try:
    sample_encoded = tokenizer.apply_chat_template(
        sample["messages"],
        tools=tools,
        tokenize=True,
        return_dict=True,
        return_assistant_tokens_mask=True,
        add_generation_prompt=False,
    )

    print("Available output keys:", sample_encoded.keys())

    if "assistant_masks" in sample_encoded:
        assistant_mask = sample_encoded["assistant_masks"]

        total_tokens = len(sample_encoded["input_ids"])
        assistant_tokens = sum(assistant_mask)

        print("\nTotal tokens:", total_tokens)
        print("Assistant tokens:", assistant_tokens)
        print(
            "Assistant token percentage:",
            round(assistant_tokens / total_tokens * 100, 2),
            "%"
        )

        print("\nAssistant-token masking is supported!")

    else:
        print("\nAssistant mask was not returned.")
        print("We will implement an alternative masking method.")

except Exception as e:
    print("Masking test encountered an issue:")
    print(type(e).__name__, str(e))

In [ ]:
# Inspect the Qwen3 chat template

print("Does the template contain {% generation %}?")

print(
    "{% generation %}" in tokenizer.chat_template
)

print("\nAssistant message formats in the dataset:")

assistant_messages = []

for message in sample["messages"]:

    if message["role"] == "assistant":

        assistant_messages.append(message)

for i, message in enumerate(assistant_messages[:5], start=1):

    print(f"\nAssistant Message {i}")
    print("Keys:", message.keys())
    print("Content:", message.get("content"))
    print("Tool calls:", message.get("tool_calls"))

In [ ]:
# Custom assistant-only masking for Qwen3

IGNORE_INDEX = -100

messages = sample["messages"]

# Tokenize the complete conversation
full_encoded = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=True,
    return_dict=True,
    add_generation_prompt=False
)

input_ids = full_encoded["input_ids"]

# Initially, ignore every token
labels = [IGNORE_INDEX] * len(input_ids)

previous_length = 0
assistant_token_count = 0

print("Checking each conversation message...\n")

# Process messages one by one
for i, message in enumerate(messages):

    current_messages = messages[:i + 1]

    # Tokenize conversation up to the current message
    current_encoded = tokenizer.apply_chat_template(
        current_messages,
        tools=tools,
        tokenize=True,
        return_dict=True,
        add_generation_prompt=False
    )

    current_ids = current_encoded["input_ids"]

    current_length = len(current_ids)

    # Verify that tokenization remains consistent
    if input_ids[:current_length] != current_ids:
        raise ValueError(
            f"Tokenization mismatch at message {i}. "
            "We need to inspect the chat template."
        )

    # Supervise assistant messages, including tool calls
    if message["role"] == "assistant":

        labels[previous_length:current_length] = (
            input_ids[previous_length:current_length]
        )

        assistant_token_count += current_length - previous_length

        print(
            f"Message {i}: Assistant | "
            f"{current_length - previous_length} tokens"
        )

    else:

        print(f"Message {i}: {message['role']} | Ignored")

    previous_length = current_length


# Calculate masking statistics
total_tokens = len(input_ids)

assistant_percentage = (
    assistant_token_count / total_tokens * 100
)

print("\n========== MASKING RESULTS ==========")

print("Total tokens:", total_tokens)
print("Assistant tokens:", assistant_token_count)
print("Ignored tokens:", total_tokens - assistant_token_count)

print(
    "Assistant-token percentage:",
    round(assistant_percentage, 2),
    "%"
)

# Verify the result
assert len(labels) == len(input_ids)

assert assistant_token_count > 0, (
    "No assistant tokens were identified!"
)

assert sum(label != IGNORE_INDEX for label in labels) == assistant_token_count

print("\nCustom assistant-only masking verified!")

In [ ]:
# Inspect Qwen3 chat-template output

print("CHAT TEMPLATE:")
print(tokenizer.chat_template)

print("\n\nFORMATTED CONVERSATION:")

formatted_text = tokenizer.apply_chat_template(
    sample["messages"],
    tools=tools,
    tokenize=False,
    add_generation_prompt=False
)

print(formatted_text)

print("\n\nSPECIAL TOKENS:")

for token in [
    "<|im_start|>",
    "<|im_end|>",
    "<|im_sep|>",
    "<|tool_call|>",
    "<|tool_response|>"
]:
    print(token, "=", tokenizer.convert_tokens_to_ids(token))

In [ ]:
import re

# 1. Format the complete conversation
formatted_text = tokenizer.apply_chat_template(
    sample["messages"],
    tools=tools,
    tokenize=False,
    add_generation_prompt=False
)

# 2. Tokenize the formatted conversation and get character offsets
encoded = tokenizer(
    formatted_text,
    add_special_tokens=False,
    return_offsets_mapping=True
)

input_ids = encoded["input_ids"]
offsets = encoded["offset_mapping"]

# 3. Verify that both tokenization methods produce identical IDs
if input_ids != full_encoded["input_ids"]:
    raise ValueError("Tokenization mismatch. Stop before training.")

print("Tokenization consistency: PASSED")

# 4. Find assistant message boundaries
pattern = r"<\|im_start\|>assistant\n(.*?)<\|im_end\|>"

assistant_spans = []

for match in re.finditer(pattern, formatted_text, re.DOTALL):
    assistant_spans.append((match.start(1), match.end(1)))

print("Assistant messages found:", len(assistant_spans))

# 5. Initialize all labels as ignored
labels = [-100] * len(input_ids)

# 6. Identify tokens belonging to assistant message content
for token_index, (token_start, token_end) in enumerate(offsets):

    if token_start == token_end:
        continue

    for span_start, span_end in assistant_spans:

        if token_start < span_end and token_end > span_start:
            labels[token_index] = input_ids[token_index]
            break

# 7. Calculate results
total_tokens = len(input_ids)

assistant_tokens = sum(
    label != -100 for label in labels
)

print("\n========== MASKING RESULTS ==========")
print("Total tokens:", total_tokens)
print("Assistant tokens:", assistant_tokens)
print("Ignored tokens:", total_tokens - assistant_tokens)

print(
    "Assistant token percentage:",
    round(assistant_tokens / total_tokens * 100, 2),
    "%"
)

assert assistant_tokens > 0, "No assistant tokens identified!"

print("\nAssistant content masking: PASSED")

In [ ]:
# Count different types of messages

from collections import Counter

message_counts = Counter()

for conversation in train_data:

    for message in conversation["messages"]:

        message_counts[message["role"]] += 1

print("Message role counts:")

for role, count in message_counts.items():

    print(f"{role}: {count:,}")

In [ ]:
# Count assistant messages with tool calls

assistant_total = 0
assistant_with_tool_calls = 0

tool_names = Counter()

for conversation in train_data:

    for message in conversation["messages"]:

        if message["role"] == "assistant":

            assistant_total += 1

            if message.get("tool_calls"):

                assistant_with_tool_calls += 1

                for tool_call in message["tool_calls"]:

                    tool_names[
                        tool_call["function"]["name"]
                    ] += 1


print("Total assistant messages:", assistant_total)

print(
    "Assistant messages with tool calls:",
    assistant_with_tool_calls
)

print("\nTool call distribution:")

for name, count in tool_names.items():

    print(f"{name}: {count:,}")

In [ ]:
import re

IGNORE_INDEX = -100

def create_assistant_labels(messages, tools, tokenizer):

    # Format the entire conversation
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=False,
        add_generation_prompt=False
    )

    # Tokenize and obtain character offsets
    encoded = tokenizer(
        formatted_text,
        add_special_tokens=False,
        return_offsets_mapping=True
    )

    input_ids = encoded["input_ids"]
    offsets = encoded["offset_mapping"]

    # Identify complete assistant messages,
    # including their message-ending token
    pattern = r"<\|im_start\|>assistant\n.*?<\|im_end\|>"

    assistant_spans = [
        (match.start(), match.end())
        for match in re.finditer(
            pattern,
            formatted_text,
            re.DOTALL
        )
    ]

    # Ignore every token initially
    labels = [IGNORE_INDEX] * len(input_ids)

    # Mark tokens overlapping assistant message spans
    for i, (start, end) in enumerate(offsets):

        if start == end:
            continue

        for span_start, span_end in assistant_spans:

            if start < span_end and end > span_start:

                labels[i] = input_ids[i]
                break

    return {
        "input_ids": input_ids,
        "attention_mask": encoded["attention_mask"],
        "labels": labels,
        "assistant_messages": len(assistant_spans)
    }


# Test on one training conversation
result = create_assistant_labels(
    sample["messages"],
    tools,
    tokenizer
)

total_tokens = len(result["input_ids"])

assistant_tokens = sum(
    label != IGNORE_INDEX
    for label in result["labels"]
)

print("Total tokens:", total_tokens)
print("Assistant messages:", result["assistant_messages"])
print("Assistant tokens:", assistant_tokens)
print("Ignored tokens:", total_tokens - assistant_tokens)

print(
    "Assistant percentage:",
    round(assistant_tokens / total_tokens * 100, 2),
    "%"
)

assert assistant_tokens > 0
assert result["assistant_messages"] == 8

print("\nAssistant-only masking test passed!")